In [0]:
%sql
SELECT * FROM cyntexa_dev.gold.daily_revenue_by_store 
ORDER BY sale_date DESC
LIMIT 20;

# Task 1: Connect Power BI Desktop to Databricks & Build a Report on a Gold Table

## Goal
Connect Power BI Desktop to a Databricks SQL warehouse (Free Edition → manual connection)
and build one report against the gold table:
`cyntexa_dev.gold.daily_revenue_by_store`

> Note: Databricks Free Edition does not support Partner Connect, so we use a
> **manual connection** with a personal access token.

---

## Architecture (simple view)

Databricks gold table → SQL Warehouse → Power BI Desktop → Report

We need 3 things from Databricks:
1. Server hostname
2. HTTP path
3. Personal access token

---

## Cell 1: Verify the gold table

```sql
SELECT * FROM cyntexa_dev.gold.daily_revenue_by_store
ORDER BY sale_date DESC
LIMIT 20;
```

Expected: rows with store_id, sale_date, total_revenue.

---

## Cell 2: Get Server Hostname & HTTP Path

1. Databricks left menu → **SQL Warehouses**.
2. Click **Serverless Starter Warehouse**.
3. Open the **Connection details** tab.
4. Copy:
   - Server hostname (example: `dbc-xxxx.cloud.databricks.com`)
   - HTTP path (example: `/sql/1.0/warehouses/xxxxx`)

---

## Cell 3: Generate a Personal Access Token

1. Profile icon (top right) → **Settings** → **Developer** → **Access tokens**.
2. **Generate new token** → name: `power-bi` → set expiry → **Generate**.
3. Copy the token immediately and save it in Notepad.
   (It is shown only once. Treat it like a password.)

---

## Cell 4: Connect from Power BI Desktop

1. Open **Power BI Desktop**.
2. Click **Get data** → search **"Azure Databricks"** → **Connect**.
3. Paste **Server hostname** and **HTTP path** → OK.
4. Sign-in window:
   - Authentication type: **Token**
   - Token: paste your access token → **Connect**.
5. Navigator: expand **cyntexa_dev → gold** → check `daily_revenue_by_store` → **Load**.

Data is now loaded into Power BI as a dataset.

---

## Cell 5: Build the Report (copy of the Databricks dashboard)

### 5.1 Bar chart — revenue by store
- Visual: **Clustered bar chart**
- X-axis: `store_id`
- Y-axis: `total_revenue` (aggregation = Sum)

### 5.2 Line chart — revenue over time
- Visual: **Line chart**
- X-axis: `sale_date`
- Y-axis: `total_revenue` (Sum)
- Legend: `store_id`

### 5.3 Slicer — store filter (like Global filter: Store-E)
- Visual: **Slicer** → Field: `store_id`
- Click **Store-E** in the slicer to filter the whole report
  (same behavior as the Databricks global filter).

---

## Cell 6: Refresh & Finish

- In Power BI: **Home → Refresh** pulls the latest data from Databricks.
- Save the file (e.g., `store_revenue_report.pbix`).


# Task 2: Unity Catalog Connection + Foreign Catalog to PostgreSQL

## Goal
Create a Unity Catalog connection to an external PostgreSQL database and a
**foreign catalog** that exposes one of its tables, so it can be queried from Databricks.

## Key Concepts (simple)

- **Lakehouse Federation** = query external data without copying it.
- **Connection** = saved credentials + address of the external database.
- **Foreign Catalog** = a read-only mirror of the external database inside Databricks.
- Data stays in PostgreSQL. Only the query travels. Results come back live.

Flow:
Your SQL → Foreign Catalog → Connection → PostgreSQL → results

## Prerequisites

- Databricks Free Edition with a **serverless** SQL warehouse (Serverless Starter is fine).
- A PostgreSQL database reachable from the internet.
  - Recommended: free cloud Postgres (Neon / Supabase).
  - Local Postgres needs an ngrok tunnel (Databricks cloud cannot reach localhost).
- PostgreSQL user + password + database name.

---

## Cell 1: Prepare sample data in PostgreSQL

```sql
CREATE TABLE customers (
  id INT PRIMARY KEY,
  name VARCHAR(50),
  city VARCHAR(50)
);

INSERT INTO customers VALUES
(1, 'Asha', 'Delhi'),
(2, 'Ravi', 'Mumbai'),
(3, 'Meena', 'Jaipur');
```

---

## Cell 2: Create the connection (GUI)

1. Databricks → **Catalog** → **External data** → **Connections** → **Create connection**.
2. Name: `postgres_conn`, Type: **PostgreSQL**.
3. Enter host, port (5432), user, password.
4. Test → Next.

---

## Cell 3: Create the foreign catalog

GUI: **Create catalog** → Type = **Foreign** → Connection = `postgres_conn`
→ Database = your Postgres DB name (e.g., `neondb`) → Test → Create.

Or via SQL:

---

## Cell 4: Query the external table

```sql
SELECT * FROM postgres_federal.public.customers;
```

Expected output: the 3 rows from PostgreSQL, live.
---

## Result
A Unity Catalog connection (`postgres_conn`) and a foreign catalog
(`postgres_federal`) expose the PostgreSQL `customers` table inside Databricks,
queryable with normal SQL — no data copied, fully read-only.

In [0]:
%sql
SELECT * FROM postgres_conn_catalog.public.customers ORDER BY id;

In [0]:
%sql
SELECT * FROM postgres_conn_catalog.public.customers ORDER BY id;

In [0]:
%sql
DESCRIBE EXTENDED postgres_conn_catalog.public.customers;

# Task 3: Lakebase — When to Use It Instead of a Delta Table

## 1. What is Lakebase? (Simple Meaning)

**Lakebase = A fully managed PostgreSQL database that lives INSIDE Databricks.**

- It is a real database for **applications** — not for analytics.
- It looks, feels, and behaves like normal PostgreSQL (same SQL, same tools like pgAdmin, psql).
- Databricks manages everything for you — no servers to set up, no patching, no scaling worries.
- It is **connected to your lakehouse** — data can flow between Delta tables and Lakebase automatically.

> Think of it this way: **Delta tables are your warehouse. Lakebase is your shop counter.** The warehouse stores everything; the counter serves customers fast.

---

## 2. The Core Idea: OLTP vs OLAP

This is the ONE concept to understand. Everything else follows from it.

| | **Delta Table (OLAP)** | **Lakebase (OLTP)** |
|---|---|---|
| Full name | Online **Analytical** Processing | Online **Transactional** Processing |
| Made for | **Analyzing** big data | **Running applications** |
| Typical question | "What was total revenue last quarter, per store?" | "Show THIS user's order status, RIGHT NOW." |
| Reads | Scans **millions of rows** at once | Fetches **one row** at a time |
| Speed style | Optimized for **big scans** | Optimized for **tiny, instant lookups** (&lt;10ms) |
| Who uses it | Analysts, data scientists, BI dashboards | Apps, websites, APIs, AI agents |
| Writes | Large batch loads | Many small, frequent writes (orders, clicks) |

**Analogy:**
- **Delta table** = A library archive. You go through thousands of files to find a pattern. Slow per item, great for research.
- **Lakebase** = A vending machine. One person presses one button, gets one item in one second. Thousands of people can use it at the same time.

---

## 3. When Would I Reach for Lakebase Instead of a Delta Table?

Reach for **Lakebase** when ALL of these are true:

1. **An application needs the data, not a human.**
   A website, mobile app, API, or AI agent is asking for data — not an analyst running a report.

2. **It must be FAST — like, really fast.**
   User-facing apps need answers in **milliseconds** (under 10ms). A Delta table query over a SQL warehouse takes seconds — fine for dashboards, terrible for a "Buy Now" button.

3. **Many users at the same time.**
   Think thousands of requests per second. Delta tables aren't built for that kind of concurrency; Lakebase handles 10,000+ queries per second.

4. **The app also WRITES data.**
   Users placing orders, updating profiles, agents saving memory. Lakebase is a real transactional database with full INSERT/UPDATE/DELETE support (foreign catalogs, remember, are read-only — Lakebase is read-AND-write).

5. **Data freshness matters, but copying is a pain.**
   Instead of building ETL pipelines to copy Delta → some other database, Lakebase can **sync with Delta tables automatically** (one-click sync, or scheduled, or continuous).

### Classic use cases
- **Customer-facing apps:** product catalogs, user profiles, shopping carts
- **AI agents:** storing agent memory/conversations (it can launch a fresh database per agent in under a second)
- **ML feature store:** serving pre-computed features to models at low latency
- **Real-time personalization:** "recommended for you" at millisecond speed

---

## 4. When to STICK with a Delta Table

Use a **Delta table** when:

| Situation | Why Delta wins |
|---|---|
| Dashboards & BI reports | SQL warehouses scan Delta fast; nobody needs 10ms |
| Big aggregations (SUM, GROUP BY over millions of rows) | This is exactly what OLAP is for |
| Data engineering pipelines (bronze → silver → gold) | Delta is the storage layer of the lakehouse |
| ML training datasets | Spark reads Delta in bulk efficiently |
| Historical/archival data | Cheap storage, ACID, time travel |

&gt; Golden rule: **Use Lakebase when an application needs to SERVE data fast. Use Delta when an analyst needs to QUERY data deep.** They are teammates, not competitors.

---

## 5. How They Work Together (the beautiful part)

```
┌────────────────────────────────────────────┐
│  DELTA TABLES (lakehouse)                  │
│  Gold layer: daily_revenue_by_store        │
│  Big, clean, analytical                    │
└───────┬────────────────────────────────────┘
        │  Auto-sync (one-click / scheduled /
        │  continuous) — no custom ETL!
        ▼
┌────────────────────────────────────────────┐
│  LAKEBASE (Postgres)                       │
│  Same data, served at > 10ms to apps        │
│  Users update rows here (orders, clicks)   │
└───────┬────────────────────────────────────┘
        │  Changes flow back as Delta tables
        ▼
┌────────────────────────────────────────────┐
│  DELTA: audit trail, history, analytics    │
└────────────────────────────────────────────┘
```

Bonus powers of Lakebase:
- **Branching:** Copy your whole database in seconds (like `git branch`) — test changes safely without touching production data.
- **Scale to zero:** Pay nothing when idle; auto-scale when busy.
- **Same governance:** Unity Catalog controls access to both Delta and Lakebase — one login, one audit trail.

---

## Quick Memory Table 

| Question | Answer |
|---|---|
| Delta table = ? | Analytics storage (OLAP) |
| Lakebase = ? | App database (OLTP), Postgres inside Databricks |
| Foreign catalog = ? | Read-only window into an EXTERNAL database |
| Lakebase vs foreign catalog? | Lakebase is Databricks' OWN database (read + write); foreign catalog links to someone else's database (read-only) |
| Latency? | Delta: seconds · Lakebase: &lt;10ms |

# Task 5: Federated Query — Delta Table JOIN Foreign Table

## Goal
Join a native Unity Catalog Delta table (`cyntexa_dev.gold.daily_revenue_by_store`)
with a foreign-catalog table (`postgres_conn_catalog.public.customers`) in ONE query,
and confirm no data was physically copied.

## Concept (simple)
- **Federated query** = one SQL query that reads from two different systems at once.
- Data is pulled live from both sides, joined in memory, shown to you.
- Nothing is stored. Nothing is copied. Query ends → everything disappears.

## Cell 1: Prepare the Postgres table (run in NEON SQL Editor)

```sql
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
  store_id VARCHAR(20) PRIMARY KEY,
  manager_name VARCHAR(50),
  city VARCHAR(50)
);

INSERT INTO customers VALUES
('Store-A', 'Asha',  'Delhi'),
('Store-B', 'Ravi',  'Mumbai'),
('Store-C', 'Meena', 'Jaipur'),
('Store-D', 'Arjun', 'Pune'),
('Store-E', 'Karan', 'Chennai');
```

Note: store_id values match the gold table so the join works.

## Cell 2: The federated join (run in DATABRICKS SQL Editor)

```sql
SELECT
  g.sale_date,
  c.store_id,
  c.manager_name,
  c.city,
  g.total_revenue,
  g.total_items_sold
FROM cyntexa_dev.gold.daily_revenue_by_store g
JOIN postgres_conn_catalog.public.customers c
  ON g.store_id = c.store_id
ORDER BY g.sale_date DESC;
```

Expected: one combined result with columns from BOTH systems.

## Cell 3: Confirm no data was copied

1. Type check:
```sql
DESCRIBE EXTENDED postgres_conn_catalog.public.customers;
-- Must show: Type = FOREIGN, Provider = postgresql
```

2. Live-update proof:
   - In Neon: UPDATE customers SET manager_name = 'Asha Verma' WHERE store_id = 'Store-A';
   - Re-run the join → new name appears instantly.
   - Conclusion: data is read live from Postgres, never copied.

3. Catalog check: no new table was created in cyntexa_dev.gold — joins don't store anything.

## Result
A single federated query joined Delta (analytics) + Postgres (operational) data with zero
ETL and zero copies. Changes at the source are visible immediately, proving live federation.

In [0]:
%sql
SELECT 
g.sale_date,
c.store_id,
c.manager_name,
c.city,
g.total_revenue,
g.total_items_sold
FROM cyntexa_dev.gold.daily_revenue_by_store g -- native Delta table
JOIN postgres_conn_catalog.public.customers c -- foreign table (live in Postgres)
ON g.store_id = c.store_id 
ORDER BY g.sale_date DESC;

In [0]:
%sql
DESCRIBE EXTENDED postgres_conn_catalog.public.customers;

# Task 6 : Prerequisites: Workspaces Setup & Organization Configuration

1. **Required Accounts:**
   * Two active Databricks accounts are required (e.g., Free Edition/Community Edition):
     * **Host Account:** Acts as the data provider sharing the gold table.
     * **Test Account:** Acts as the recipient consuming the shared data.

2. **Configure Host Account Organization Name:**
   * Log in to your **Host Account**.
   * Go to **Catalog Explorer** -> Click **OpenSharing** (top right).
   * Click on the **Organization Name** dropdown (located directly to the left of **Settings** at the top right) -> Select **Edit organization name**.
   * Update the organization name to a recognizable label (e.g., `mohit_main`) and click **Save**.

3. **Configure Test Account Organization Name:**
   * Log in to your **Test Account**.
   * Navigate to **Catalog Explorer** -> **OpenSharing**.
   * Click on the **Organization Name** dropdown (to the left of **Settings**) -> Select **Edit organization name**.
   * Update the organization name to a distinct label (e.g., `mohit_test`) and click **Save**.

# How I Enabled and Shared My Gold Table using Delta Sharing

## Step 1: Check Metastore Status & Enable External OpenSharing
1. Go to **Catalog Explorer** and click the gear (Settings) icon at the top left.
2. Under **Metastore**, click on your metastore (`metastore_aws_us_east_2`) to check its status (it will show `External OpenSharing: disabled`).
3. Go back to **Catalog Explorer** -> Click **Share** at the top right -> Select **OpenSharing**.
4. In the OpenSharing page, click **Settings** (gear icon at the top right).
5. Toggle **External OpenSharing** to **Enabled**, set your **Organization name** (e.g., `mohit_main`), and click **Save**.
6. Go back to your **Metastore settings** to verify that it now shows `External OpenSharing: enabled`.

---

# Step 2: Create Share, Add Table, and Setup Recipient (Host Account)

1. Open **Catalog Explorer** -> Click **Share** (top right) -> Select **OpenSharing**.
2. Go to the **Shared by me** tab and click the **Share data** button.
3. **Create Share:**
   * Enter **Share name** (e.g., `gold_revenue_share`).
   * Click **Save and continue**.
4. **Add Data Assets:**
   * Expand your catalog and schema (e.g., `cyntexa_dev` -> `gold`).
   * Select your table (e.g., `daily_revenue_by_store`).
   * Click **Save and continue**.
5. **Add Recipient:**
   * On the **Add recipients** step, click **Select a recipient** dropdown.
   * If recipient is not created, click **+ Create new recipient**:
     * Open your **Test Account** -> Go to **Catalog** -> **OpenSharing** -> Copy the **Sharing ID** from top right.
     * Paste the **Sharing ID** into the Host Account recipient creation box, enter recipient name (e.g., `mohit_test`), and save.
   * Select `mohit_test` from the dropdown list.
6. Click **Share data** at the bottom right.
7. You will now see your **Gold Table** successfully listed under the created Share (`gold_revenue_share`).

---

# Step 3: Configure Metastore Access Requests and Request OpenSharing Permissions (Test Account & Host Account)

1. **Verify Share Status (Host Account):**
   * Even though the table and recipient (`mohit_test`) are added in the Host Account's Share (`gold_revenue_share`), the shared data does not appear in the Test Account due to missing metastore access permissions.

2. **Configure Access Requests (Test Account):**
   * Open **Catalog Explorer** -> Click the **Gear icon** (top right) -> Select **Metastore** (`metastore_aws_us_east_2`).
   * In the **Details** tab, locate **Access requests** (currently showing *Partially enabled*).
   * Click the **Pencil (edit)** icon next to *Partially enabled*.
   * In the popup dialog under **Email**, enter the Host Account administrator's email address.
   * Click **Save**.

3. **Request OpenSharing Permissions (Test Account):**
   * Go back to **Catalog Explorer** -> **OpenSharing** -> **Shared with me** tab.
   * Click the **Request OpenSharing permissions** button.
   * In the request modal:
     * Select/Verify the **Principals** email address.
     * Ensure **USE PROVIDER** permission is checked.
     * Click **Request**.

4. **Approve Request (Host Account):**
   * The Host Account owner will receive an email titled **"A new Unity Catalog access request is awaiting your review"**.
   * Click **Open permission settings** inside the email and accept the request to enable sharing access.

---

# Step 4: Mount Shared Data to Catalog and Query Table (Test Account)

1. **Access Shared Provider Data:**
   * Open **Catalog Explorer** -> **OpenSharing** -> **Shared with me** tab.
   * You will now see the provider (e.g., `mohit_main`) listed along with `databricks_system_tables`.
   * Click on the host organization name (`mohit_main`).

2. **Mount Share to Catalog:**
   * Locate the shared data (`gold_revenue_share`).
   * Click the **Mount to catalog** button next to it.
   * In the popup modal:
     * Select **Create a new catalog** (or choose *Mount to existing catalog*).
     * Enter a **Catalog name** (e.g., `test`).
     * Click **Create**.

3. **Verify and Access Data:**
   * Go to the left navigation panel under **Catalog Explorer**.
   * Locate the newly created catalog (`test`) featuring the shared catalog icon.
   * Expand `test` -> `gold` -> `Tables` to view `daily_revenue_by_store`.
   * Click the table to view its schema details (`sale_date`, `store_id`, `total_revenue`, `total_items_sold`).
   * The shared table is now active and ready for querying.

# Supported Content Types in Databricks-to-Databricks Open Sharing

When you share data between two Databricks workspaces (provider → recipient),
you can put these 3 types of content inside a share:

---

## 1. Tables ✅

**What it is:** Normal data tables with rows and columns (like your gold table).

- The data lives in the **provider's storage**.
- The recipient sees it and can run `SELECT` queries on it — **read-only**.
- When the provider updates the data, the recipient sees the new data instantly (live data, no copies).

**Example shared:** `cyntexa_dev.gold.daily_revenue_by_store`

> Think of it like: giving someone a **live camera feed** of your table — they can watch, but not touch.

---

## 2. Views ✅

**What it is:** A saved query that looks like a table, but has no data of its own.

- A view is just a **recipe**: "take the table, filter it, hide some columns, then show."
- When the recipient queries the view, the recipe runs and they see the result.
- **Dynamic views** are special — they can hide rows or mask columns per recipient (security!).

**Example:** Share a view that shows only `Store-E` data, or hides the `total_revenue` column.

> Think of it like: a **filtered window** into your table. You decide exactly what the other side is allowed to see.

---

## 3. Volumes ✅

**What it is:** A folder of **files** stored in Unity Catalog (not a table).

- Tables hold structured data (rows/columns).
- Volumes hold **anything as files**: PDFs, images, videos, audio, text files, JSON, CSV...
- The recipient can **download/read** the files — again, read-only.

**Example:** Share a volume containing product images or monthly report PDFs.

> Think of it like: a **shared Google Drive folder** — files, not tables.

---

## Quick Comparison Table

| Content Type | What it holds | Shared as | Recipient can |
|---|---|---|---|
| **Table** | Rows & columns (structured data) | Live data | Query (SELECT only) |
| **View** | A saved/filtered query | Live query result | Query (SELECT only) |
| **Volume** | Files (PDFs, images, etc.) | Files | Read & download |


# Task 7: Import an Existing Power BI Report into a Databricks Dashboard

## Goal
Import an existing Power BI report into Databricks (AI/BI) Dashboards using the
"Import from Power BI" feature, and document what did not translate cleanly.

## Key Facts
- The importer accepts **.pbit** (Power BI template) files only — not .pbix.
- (Beta: PBIR / enhanced report format is also accepted.)
- Genie Code reads the .pbit and auto-builds an AI/BI dashboard + metric views.

## Cell 1: Export the .pbit from Power BI Desktop
- Open the .pbix (Task 1 report).
- File → Export → Power BI template (.pbit) → save as store_revenue_template.pbit.

## Cell 2: Import into Databricks
- Dashboards → Create dashboard → ▼ → Import from Power BI.
- Attach the .pbit file (must be &lt; 100 MB).
- Genie Code opens and builds the dashboard from the file.

## Cell 3: Connect the data source
- When prompted, choose the serverless SQL warehouse.
- Map to cyntexa_dev.gold.daily_revenue_by_store.

## Cell 4: Review the generated dashboard
- Compare with the original Power BI report.
- Core visuals (bar, line, store filter) should match.

## Cell 5: What did NOT translate cleanly

| Power BI feature | Result in Databricks | Notes |
|---|---|---|
| Custom theme (Bloom purple) | Lost | Default dashboard colors |
| DAX measures | Converted to metric views (SQL) | Verify the logic manually |
| Custom/marketplace visuals | Replaced by built-in types | Closest match only |
| Bookmarks / drill-through / tooltips | Not supported | Removed |
| Slicers | Became simple filters | Less styling control |
| Fine formatting / sorting | Partial | May need manual fixes |

## Result
The Power BI report was imported via .pbit and rebuilt as an AI/BI dashboard by
Genie Code. Data and core visuals translated cleanly; themes, DAX, custom visuals,
and advanced interactivity did not.

## One-line summary
> Power BI reports migrate to Databricks dashboards via .pbit templates: data,
> charts, and filters translate; theming, DAX logic, and Power BI-only
> interactive features (bookmarks, tooltips, custom visuals) do not.

# Task 8: Data-Sharing Decision Matrix for Cyntexa

## Goal
For any partner scenario, decide which sharing method to use:
**Databricks-to-Databricks** OR **Databricks-to-Open** — and explain why.

---

## 1. First, Understand the 2 Sharing Methods (Simple)

### Method A: Databricks-to-Databricks (D2D)

**What it is:** Both companies use Databricks. Data flows from one Unity Catalog
to another Unity Catalog — like two Databricks workspaces "shaking hands" securely.

- The partner **mounts** your share as a normal catalog in their workspace.
- They query it with normal SQL. Data is live, read-only, never copied.
- Works through OAuth (secure identity check) — no passwords or files to manage.

### Method B: Databricks-to-Open (Open Protocol)

**What it is:** The partner does **NOT** use Databricks. They use any tool that
understands the open Delta Sharing protocol:

- Power BI, Tableau, Excel
- Python (pandas / Spark), Trino, custom apps (via REST API)

**How it works:**
1. You (provider) create a **recipient** and download a small **credential file**
   (contains an endpoint URL + a token).
2. You send this file to the partner (email/Slack — your choice).
3. Partner's tool uses the file to fetch data — files are downloaded directly
   from cloud storage using temporary pre-signed URLs.

> Simple analogy:
> - **D2D** = Two Databricks users sharing a folder inside one big secure building.
> - **Open** = Handing an external visitor a temporary guest pass (credential file)
>   so they can pick up files at the reception desk.

---

## 2. What Each Method Can Share (Asset Support)

| Asset type | Databricks-to-Databricks | Databricks-to-Open |
|---|---|---|
| Tables (Delta, Parquet, etc.) | ✅ Yes | ✅ Yes |
| Volumes (files) | ✅ Yes | ✅ Yes |
| Views (standard) | ✅ Yes | ❌ No |
| Dynamic views (row/column masking) | ✅ Yes | ❌ No |
| Notebooks | ✅ Yes | ❌ No |
| Registered AI/ML models | ✅ Yes | ❌ No |
| Materialized views / streaming tables | ✅ Yes | ❌ No |
| Foreign tables | ✅ Yes | ❌ No |

**Why the difference?** Open-protocol clients only know how to read *files* and
*tables*. Things like views, notebooks, and models need a live Databricks/Unity
Catalog brain on the receiving side — which only exists in D2D.

---

## 3. The Decision Matrix 

Ask these 2 questions about the partner:

| Question 1: Does the partner have their own Databricks workspace? | Question 2: What do they need? | → Decision | Why |
|---|---|---|---|
| **YES** | Tables only | **D2D** | Richest security, live data, easy revocation |
| **YES** | Views / notebooks / AI models / volumes | **D2D** (only option) | These assets are NOT supported by the open protocol |
| **NO** | Tables or files (volumes) | **Databricks-to-Open** | Only option — send a credential file, they use Power BI/Python/etc. |
| **NO** | Views / notebooks / AI models | ⚠️ **Problem!** | Not possible directly. Workaround: share the *underlying tables* via Open, and let the partner rebuild views/models in their own tools. |

### 4 Ready-Made Partner Scenarios for Cyntexa

| Partner scenario | Right choice | Reason |
|---|---|---|
| Partner agency also runs Databricks; wants your gold table + a masked view (hide revenue column) | **D2D** | Dynamic views only work D2D |
| Client uses only Power BI, wants the sales table for their reports | **Open** | They have no Databricks; Power BI reads via credential file |
| ML partner wants your trained model + feature tables | **D2D** | Registered models are D2D-only |
| Startup wants raw CSV-style files for a Python pipeline | **Open** | Simplest for them; Python delta-sharing connector works |

---

## 4. Decision Rules 

1. **Databricks on both sides → always choose D2D.** It is more secure
   (OAuth, no file tokens), supports more asset types, and gives full audit trails.
2. **No Databricks on the other side → Databricks-to-Open with a credential file.**
   Works with any tool, but limited to tables and volumes.
3. **If they need views/models/notebooks but have no Databricks →**
   share the base tables openly and let them rebuild the logic on their side.

---

## One-line Summary

> Use **Databricks-to-Databricks** when the partner has a Databricks workspace
> (secure OAuth handshake, supports everything: tables, views, volumes, notebooks,
> AI models). Use **Databricks-to-Open** when they don't (credential file for
> Power BI/Python/etc., but only tables and volumes can be shared).

# Task 9 : Query Federation vs Nightly Copy Pipeline for Postgres

## The Question
When should we query Postgres **directly at runtime (federation)** vs **copying data into Databricks every night**?

---

## Quick Definitions

| Term | Meaning |
|------|---------|
| **Query Federation** | Databricks connects to Postgres live and pulls data only when a query runs. No data is copied. |
| **Nightly Copy Pipeline** | Data is copied from Postgres into Databricks once every night. Queries run fast on the local copy. |

---

## How Do They Compare?

| Factor | Federation (Query Live) | Nightly Copy (ETL) |
|--------|------------------------|---------------------|
| Data freshness | Real-time / always latest | Up to 24 hours old |
| Query speed | Slower (data travels over network) | Very fast (data is local) |
| Load on Postgres | High during heavy queries | Low (reads once at night) |
| Setup effort | Low (just a connection) | Higher (need a pipeline + storage) |
| Postgres can go down? | Queries fail | Queries still work on old copy |

---

## When Does Federation Make Sense?

Federation is a good choice when **most of these are true**:

1. **Freshness matters a lot** — the business needs data that is minutes old, not a day old
   - Example: checking if an order was just placed, current stock levels, live prices

2. **Query volume is LOW** — only a few users run these queries, a few times a day
   - Light traffic won't hurt Postgres

3. **The data set is SMALL** — like lookup tables, reference data, a few thousand rows
   - Small transfers are fast and cheap

4. **You want a quick start** — no time to build an ETL pipeline, just connect and go

---

## When Should You Use the Nightly Copy Instead?

1. **Data freshness is NOT critical** — yesterday's data is good enough
   - Example: daily sales reports, monthly trends

2. **Query volume is HIGH** — many users, many queries, heavy analytics
   - This will overload Postgres and slow down the live application using it

3. **Large data / heavy joins** — scanning millions of rows repeatedly
   - The network becomes a bottleneck

4. **You need reliability** — if Postgres is down, your reports still work on the local copy

---

## The Simple Rule of Thumb

> **Federation = fresh but slow and gentle use.**
> **Nightly copy = slightly stale but fast and safe for heavy use.**

- Need **live data + few small queries** → use **federation**
- Need **heavy analytics + data can be a day old** → use **nightly copy**

---

## Bonus: A Common Best Practice

Many teams use a **hybrid** approach:
- Copy the big tables nightly (heavy reporting)
- Federate the small, live tables (real-time lookups)

This gives you speed for analytics and freshness where it actually matters.

# LTAP Architecture: Real-Time Inventory App for Cyntexa

## The Question
Design a system where:
- **OLTP writes** (fast, small, real-time transactions) happen in **Lakebase** (Postgres-based operational database)
- **OLAP analytics** (big, complex reporting) happen in the **Lakehouse**
- We must define **what syncs where** and **who owns each side**

---

## Quick Definitions

| Term | Meaning |
|------|---------|
| **LTAP** | Lakehouse Transactional **and** Analytical Processing — one platform handling both fast writes and big analytics |
| **OLTP** | Day-to-day operations (add item, sell item, update stock) — many small fast writes |
| **OLAP** | Analytics and reporting (trends, dashboards, forecasts) — few big read queries |
| **Lakebase** | Databricks' Postgres engine — handles the OLTP side |
| **Lakehouse** | Delta tables in Databricks — handles the OLAP side |

---

## The Proposed Architecture


In [0]:
displayHTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true});</script>
<div class="mermaid">
flowchart TD
    A["<b>USERS / APP</b><br/>Warehouse staff, mobile app, web UI"] -->|"small fast reads/writes"| B["<b>LAKEBASE (OLTP)</b> — Lakebase / Postgres<br/>• update stock • place order • reserve item<br/><i>Single source of truth for CURRENT state</i>"]
    B -->|"CDC stream (Auto Loader / Lakebase replication → Delta tables)"| C["<b>LAKEHOUSE (OLAP)</b> — Delta tables<br/>• inventory trends • sales dashboards<br/>• demand forecasting • slow-mover reports<br/><i>Historical + analytical copy of the data</i>"]
</div>
""")


---

## What Syncs Where?

| Data Flow | Direction | How | Why |
|-----------|-----------|-----|-----|
| All writes | **App → Lakebase only** | Direct SQL writes | Single writer = no conflicts, real-time truth |
| Changes stream | **Lakebase → Lakehouse** | CDC (Change Data Capture) — every insert/update/delete copied as it happens | Analytics always near-real-time (seconds to minutes lag) |
| Analytics results | **Lakehouse → App (read-only)** | App queries a "recommendations" or "forecast" table in the Lakehouse | Read-only, no write conflicts |
| **No direct app writes to the Lakehouse** | ❌ | — | Prevents two systems disagreeing about current state |

---

## Why This Design? (Keeping Q9 in Mind)

- This is the **nightly copy idea from Q9, but continuous**: instead of one batch a day, a **CDC stream** keeps the Lakehouse updated in near-real-time
- All writes go **one place (Lakebase)** — so there is never confusion about "which system is right?"
- Federation-style queries (Q9) are replaced by a **synced copy**, because analytics volumes are too heavy to run against Postgres live

---

## Who Owns What?

| Area | Owner | Responsibilities |
|------|-------|------------------|
| **Lakebase (OLTP)** | **Application/Backend team** | Schema design, API writes, indexes, uptime of the app, data correctness |
| **CDC sync pipeline** | **Data Platform team** | The replication job, monitoring lag, fixing broken syncs |
| **Lakehouse (OLAP)** | **Data/Analytics team** | Delta table design, dashboards, forecasts, data quality checks |
| **SLAs & alerts** | **Shared** | E.g., "sync lag must stay under 5 minutes" — app team alerts on app issues, data team alerts on sync issues |

---

## Simple Rule of Thumb

> **Lakebase = the beating heart (current truth).**
> **Lakehouse = the memory (history + analytics).**
> **Data flows one way: writes go in, changes flow out. Never write to both.**

This gives Cyntexa real-time inventory operations **and** powerful analytics, without the two systems ever fighting over the same data.